In [ ]:
import os
import glob
import cv2
import numpy as np
import concurrent.futures
from tqdm import tqdm
import time
from ultralytics import YOLO

# ===================================================================
# 1. ビルディングブロックとなるクラス定義
#    - HSVベースの処理で使用します
# ===================================================================

class MaskCreator:
    """HSV色空間に基づいて画像からマスクを生成するクラス。"""
    def __init__(self, lower_hsv=None, upper_hsv=None):
        self.lower_hsv = lower_hsv if lower_hsv is not None else np.array([20, 100, 100])
        self.upper_hsv = upper_hsv if upper_hsv is not None else np.array([255, 255, 255])

    def create_mask(self, image_bgr):
        hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, self.lower_hsv, self.upper_hsv)
        return cv2.bitwise_not(mask)

class BB_crop:
    """マスクを元に元画像をクロップして保存するクラス。"""
    def __init__(self, padding=30, save_dir="./cropped_images"):
        self.padding = padding
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)

    def save(self, image_bgr, mask, filename="cropped.jpg"):
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            raise ValueError("No contours found in mask")

        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)

        height, width = image_bgr.shape[:2]
        x1 = max(x - self.padding, 0)
        y1 = max(y - self.padding, 0)
        x2 = min(x + w + self.padding, width)
        y2 = min(y + h + self.padding, height)

        cropped = image_bgr[y1:y2, x1:x2]
        save_path = os.path.join(self.save_dir, filename)
        cv2.imwrite(save_path, cropped)
        return save_path

class BB_mask:
    """マスク自体をクロップして保存するクラス。"""
    def __init__(self, padding=30, save_dir="./cropped_masks"):
        self.padding = padding
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)

    def save(self, mask, filename="cropped_mask.jpg"):
        if len(mask.shape) == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            raise ValueError("No contours found in mask")

        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)

        height, width = mask.shape[:2]
        x1 = max(x - self.padding, 0)
        y1 = max(y - self.padding, 0)
        x2 = min(x + w + self.padding, width)
        y2 = min(y + h + self.padding, height)

        cropped_mask = mask[y1:y2, x1:x2]
        save_path = os.path.join(self.save_dir, filename)
        cv2.imwrite(save_path, cropped_mask)
        return save_path

# ===================================================================
# 2. 各プロセスから呼び出されるヘルパー関数
# ===================================================================

def process_image_hsv(args):
    """HSVベースのクロップとマスク保存を単一の画像に対して行う関数。"""
    image_path, crop_save_dir, mask_save_dir, padding = args
    try:
        filename = os.path.basename(image_path)
        image = cv2.imread(image_path)
        if image is None:
            return f"読み込み失敗: {filename}"

        mask_creator = MaskCreator()
        mask = mask_creator.create_mask(image)

        image_saver = BB_crop(padding=padding, save_dir=crop_save_dir)
        mask_saver = BB_mask(padding=padding, save_dir=mask_save_dir)

        image_saver.save(image, mask, filename="cropped_" + filename)
        mask_saver.save(mask, filename="mask_" + filename)
        return f"処理完了: {filename}"
    except Exception as e:
        return f"エラー ({os.path.basename(image_path)}): {e}"

def process_image_yolo(args):
    """YOLOベースの物体検出とセグメンテーションマスクの保存を単一の画像に対して行う関数。"""
    image_path, save_dir, crop_model_path, seg_model_path = args
    try:
        cropper = YOLO(crop_model_path)
        segmenter = YOLO(seg_model_path)

        image = cv2.imread(image_path)
        if image is None:
            return f"読み込み失敗: {os.path.basename(image_path)}"

        results = cropper(image_path)[0]
        base_name = os.path.splitext(os.path.basename(image_path))[0]

        for i, box in enumerate(results.boxes.xyxy):
            x1, y1, x2, y2 = map(int, box)
            cropped_img = image[y1:y2, x1:x2]

            seg_results = segmenter(cropped_img)[0]
            if seg_results.masks is not None:
                masks = seg_results.masks.data.cpu().numpy()
                for j, mask_data in enumerate(masks):
                    bin_mask = (mask_data * 255).astype('uint8')
                    save_path = os.path.join(save_dir, f"mask_{base_name}_crop{i}_seg{j}.png")
                    cv2.imwrite(save_path, bin_mask)
        return f"処理完了: {os.path.basename(image_path)}"
    except Exception as e:
        return f"エラー ({os.path.basename(image_path)}): {e}"

# ===================================================================
# 3. フォルダ単位で処理を実行するクラス
# ===================================================================

class HsvFolderProcessor:
    """指定されたフォルダ内の全画像にHSVベースの処理を並列実行するクラス。"""
    def __init__(self, input_dir, crop_save_dir, mask_save_dir, padding=30):
        self.input_dir = input_dir
        self.crop_save_dir = crop_save_dir
        self.mask_save_dir = mask_save_dir
        self.padding = padding
        os.makedirs(self.crop_save_dir, exist_ok=True)
        os.makedirs(self.mask_save_dir, exist_ok=True)

    def run(self):
        image_paths = []
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            image_paths.extend(glob.glob(os.path.join(self.input_dir, ext.lower())))
            image_paths.extend(glob.glob(os.path.join(self.input_dir, ext.upper())))
        
        if not image_paths:
            print(f"⚠️ {self.input_dir} に画像が見つかりません。")
            return

        print(f"📂 {len(image_paths)} 枚の画像をHSVベースで並列処理します。")
        tasks = [(path, self.crop_save_dir, self.mask_save_dir, self.padding) for path in image_paths]

        with concurrent.futures.ProcessPoolExecutor() as executor:
            list(tqdm(executor.map(process_image_hsv, tasks), total=len(image_paths), desc="HSV処理中"))
        print("✅ HSVベースの処理が完了しました。")

class YoloFolderProcessor:
    """指定されたフォルダ内の全画像にYOLOベースの処理を並列実行するクラス。"""
    def __init__(self, input_dir, save_dir, crop_model_path, seg_model_path):
        self.input_dir = input_dir
        self.save_dir = save_dir
        self.crop_model_path = crop_model_path
        self.seg_model_path = seg_model_path
        os.makedirs(self.save_dir, exist_ok=True)

    def run(self):
        image_paths = []
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
            image_paths.extend(glob.glob(os.path.join(self.input_dir, ext.lower())))
            image_paths.extend(glob.glob(os.path.join(self.input_dir, ext.upper())))

        if not image_paths:
            print(f"⚠️ {self.input_dir} に画像が見つかりません。")
            return

        print(f"📂 {len(image_paths)} 枚の画像をYOLOベースで並列処理します。")
        tasks = [(path, self.save_dir, self.crop_model_path, self.seg_model_path) for path in image_paths]

        with concurrent.futures.ProcessPoolExecutor() as executor:
            list(tqdm(executor.map(process_image_yolo, tasks), total=len(image_paths), desc="YOLO処理中"))
        print("✅ YOLOベースの処理が完了しました。")



In [ ]:
# ===================================================================
# 4. 実行ブロック
# ===================================================================
if __name__ == '__main__':
    # --- ユーザー設定 ---
    # 処理したいワークフローを選択してください ('HSV' または 'YOLO')
    WORKFLOW_TO_RUN = 'HSV'

    processor = None

    # --- HSVワークフロー用設定 ---
    if WORKFLOW_TO_RUN == 'HSV':
        input_directory = "/home/data/0827_seido/org/A"
        cropped_image_output_dir = "/home/data/1021_fasttest/cropped_images"
        mask_output_dir = "/home/data/1021_fasttest/masks"
        processor = HsvFolderProcessor(
            input_dir=input_directory,
            crop_save_dir=cropped_image_output_dir,
            mask_save_dir=mask_output_dir,
            padding=30
        )

    # --- YOLOワークフロー用設定 ---
    elif WORKFLOW_TO_RUN == 'YOLO':
        input_directory = "/home/data/maesyori_img"
        yolo_output_dir = "/home/test_hozon/yolo_masks"
        # ご自身の環境に合わせてモデルパスを書き換えてください
        cropper_model_path = "/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt"
        segmenter_model_path = "/home/YOLO/-327_seg/datasets/train2/weights/best.pt"
        processor = YoloFolderProcessor(
            input_dir=input_directory,
            save_dir=yolo_output_dir,
            crop_model_path=cropper_model_path,
            seg_model_path=segmenter_model_path
        )
    
    else:
        print(f"エラー: WORKFLOW_TO_RUN は 'HSV' または 'YOLO' に設定してください。")

    # --- 処理の実行 ---
    if processor:
        start_time = time.perf_counter()
        processor.run()
        end_time = time.perf_counter()
        print(f"🚀 全体の実行時間: {end_time - start_time:.2f} 秒")

